# LoPy 影视级工作流 - Google Colab 快速运行笔记本

本笔记本支持在 **Google Colab** 环境下极速运行 MiniMax H3 (Ref2VA) 视频生成模型。
- 建议运行时硬件：**L4 (24GB Ada Lovelace)** 或 **A100 (40GB/80GB)**。
- 相对于老旧架构（如 T4），L4/A100 具备原生 FP8 Tensor Cores，单镜推理提速 **50~100 倍**（3 秒视频耗时约 1.5~2.5 分钟）。
- 本环境遵循商业友好协议，无非商用授权限制。

In [ ]:
# [步骤 1] 检查 GPU 分配情况
!nvidia-smi

### [步骤 2] （可选）挂载 Google Drive
如果您的 Google Drive 中已存有 H3 模型权重（约 42GB），挂载后可直接复用，免去每次开机重复下载的等待。

In [ ]:
from google.colab import drive
import os

USE_DRIVE = False # 如果希望挂载 Google Drive，请改为 True
if USE_DRIVE:
    drive.mount('/content/drive')
    print("Google Drive 挂载成功！")
else:
    print("跳过 Google Drive 挂载，将使用 Colab 临时磁盘。")

### [步骤 3] 克隆仓库并运行一键配置环境
运行 `cloud/colab_setup.sh` 脚本配置 ComfyUI、Veda Sparse Attention 节点及系统依赖。

In [ ]:
%cd /content
!rm -rf lopy-workflow-colab
!git clone https://github.com/your-username/lopy-workflow-colab.git # 若有私有仓库可替换或上传
%cd /content/lopy-workflow-colab
!bash cloud/colab_setup.sh

### [步骤 4] 准备模型权重
如果是首次运行且未挂载 Google Drive，可运行下列脚本多线程高速拉取 H3 权重（Colab 机房通常 100~200MB/s，下载 42GB 约需 5~8 分钟）。

In [ ]:
# 检查本地或挂载的模型目录
!bash setup/download_stage2_stage3_models.sh

### [步骤 5] 后台启动 ComfyUI 推理服务

In [ ]:
import subprocess, time, urllib.request

# 启动 ComfyUI
cmd = (
    "python3 /content/ComfyUI/main.py "
    "--listen 127.0.0.1 --port 8188 "
    "--input-directory /content/lopy-workflow-colab/runtime/inputs "
    "--output-directory /content/lopy-workflow-colab/runtime/outputs "
    "--lowvram"
)

log_file = open("/content/comfy_server.log", "w")
proc = subprocess.Popen(cmd, shell=True, stdout=log_file, stderr=subprocess.STDOUT)
print("正在启动 ComfyUI 引擎...")

# 等待服务响应
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        if r.status == 200:
            print("✅ ComfyUI 启动就绪！")
            break
    except Exception:
        time.sleep(2)
else:
    print("ComfyUI 启动中，请查看 /content/comfy_server.log 日志。")

### [步骤 6] 运行分镜视频测试
生成 9:16 竖屏 (768×1344) 3秒测试镜头并合成音视频。

In [ ]:
!python3 tools/h3_shot.py tests/video_test/s1_test.json

### [步骤 7] 成片预览与下载

In [ ]:
from IPython.display import HTML
from base64 import b64encode
from pathlib import Path

mp4_path = Path("tests/video_test/out/clip.mp4")
if mp4_path.exists():
    mp4 = mp4_path.read_bytes()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=400 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("未找到生成的 clip.mp4，请检查生成日志。")